# Colab T4 SSH → Cloudflare tunnel (polished)

Run every cell in order. At the end Colab prints a **trycloudflare SSH link**.
Paste that link (and the printed `ssh ...` command) back to the OpenHands agent.
It will connect, verify `nvidia-smi`, clone the repo and run the GPU benchmark.

- Runtime: **T4 GPU** → Menu "Runtime" > "Change runtime type" > T4 GPU.
- Safe to re-run after a session expires; Drive persisted config is reused if mounted.

### 1. Sanity check (GPU + OS)

In [ ]:
import platform, subprocess, sys, os, shutil
print("python", sys.version.split()[0])
print("os  ", platform.platform())

def sh(c):
    p = subprocess.run(c, shell=True, capture_output=True, text=True)
    return p.stdout.strip() + (("[err] " + p.stderr.strip()) if p.returncode else "")

nv = shutil.which("nvidia-smi") or next(
    (x for x in ("/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi") if os.path.exists(x)), None)

if nv:
    print(sh('"' + nv + '" --query-gpu=name,memory.total --format=csv'))
else:
    print("!! nvidia-smi NOT found -> you are on a CPU/TPU runtime (no GPU).")
    print(">> Fix: Menu > Runtime > Change runtime type > Hardware accelerator: T4 GPU")
    print(">> The runtime restarts. Then press Ctrl+F9 / Runtime > Run all again.")
print("user:", sh("whoami"))


### 2. (Optional) Mount Drive for persistence across sessions
If the prompt appears, authorise **the same Google account** the notebook runs under.

In [ ]:
import subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")
except Exception as e:
    print("Drive mount skipped:", type(e).__name__)

### 3. Install the SSH server + tools

In [ ]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openssh-server tmux htop >/dev/null 2>&1
print("ok")

### 4. Configure sshd: root login with keys AND password

In [ ]:
from pathlib import Path
Path("/etc/ssh").mkdir(parents=True, exist_ok=True)
Path("/etc/ssh/sshd_config").write_text('Port 22\nPermitRootLogin yes\nPasswordAuthentication yes\nPubkeyAuthentication yes\nAuthorizedKeysFile .ssh/authorized_keys\nChallengeResponseAuthentication no\nUsePAM yes\nX11Forwarding yes\nPrintMotd no\nClientAliveInterval 120\nAcceptEnv LANG LC_*\nSubsystem sftp internal-sftp\n')
print("sshd_config written")
import subprocess
# make sure root password is the known one: !123
r = subprocess.run(["passwd"], input=b"!123\n!123\n", capture_output=True)
print("passwd rc:", r.returncode)

### 5. Authorise OpenHands SSH keys

In [ ]:
from pathlib import Path
Path("/root/.ssh").mkdir(mode=0o700, parents=True, exist_ok=True)
auth = Path("/root/.ssh/authorized_keys")
auth.touch(mode=0o600, exist_ok=True)
text = auth.read_text()
for pub in ['ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIEunVTqNAmyvlRh2ysy5hlkBQTyd5rZwwixgSUwpWZ0M openhands@all-hands.dev', 'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIPZ8MSR/0P1QlZqPiV8YKcTg4SvxLyysCnpgCoCbCOBT openhands@all-hands.dev', 'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIMr78cWgUocOWImAhxBZ2NDYdblaTWUYa/30H0yVesOX openhands-colab-t4']:
    if pub not in text:
        text = text.rstrip() + "\n" + pub + "\n"
auth.write_text(text)
auth.chmod(0o600)
print("authorized_keys entries:", len(auth.read_text().splitlines()))

### 6. Start sshd + verify listening on :22

In [ ]:
!service ssh restart
!ss -tlnp | grep ':22 ' || echo '(not listening on 22 yet)'
print("sshd running:", subprocess.run(["pgrep", "-a", "sshd"], capture_output=True, text=True).stdout or "no")

### 7. Launch the Cloudflare tunnel (this prints your link!)

In [ ]:
!pip install -q --upgrade colab_ssh
import contextlib, io, re

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    from colab_ssh import launch_ssh_cloudflared
    launch_ssh_cloudflared(password="!123", verbose=True)
out = buf.getvalue()
print(out)

hosts = sorted(set(re.findall(r"[A-Za-z0-9-]+\.trycloudflare\.com", out)))
print("\n========== COPY THIS BACK TO THE AGENT ==========")
print("HOSTNAME:", hosts[0] if hosts else "(none found)")
if hosts:
    print("SSH COMMAND:  ssh root@" + hosts[0])
    print("Also saved to /root/colab_ssh_host.txt")
    from pathlib import Path
    Path("/root/colab_ssh_host.txt").write_text(hosts[0] + "\n")
print("===================================================")


### 8. (Optional) Keep runtime alive + watch the tunnel
The next cell keeps the session busy so Google doesn't idle it out. It won't block you from seeing the link above.

In [ ]:
import time
from pathlib import Path
print("Keep-alive (Ctrl+C / refresh to stop). Press ▶ again to resume after disconnect.")
try:
    while True:
        Path('/root/.keepalive').touch()
        time.sleep(60)
except KeyboardInterrupt:
    print("stopped")